# Day 09 · 멀티에이전트 오케스트레이션과 웹개발

한 에이전트로 안 되는 규모를 **나누고·병렬로·검증하며** 해낸다. 오케스트레이션 패턴과 루프 엔지니어링을 코드로 익히고, 여러 에이전트로 실제 **Next.js 앱**을 만든다.

- Lab 1~2는 **NVIDIA Qwen**으로 패턴을 코드로 체험 (여기선 순차 데모).
- Lab 3(웹앱 빌드)은 **로컬 Claude Code**에서 서브에이전트/오케스트레이션으로 진행.

## Lab 0 · 셋업

패키지 설치 → LLM 연결 → 한 번의 호출을 '에이전트 하나'로 쓰는 `ask` 헬퍼.

In [ ]:
%pip install -q openai

In [ ]:
# 오케스트레이션·루프 데모에서 LLM으로 Qwen을 부른다. 토큰은 입력창/환경변수 (하드코딩 금지).
import os, getpass, json
from openai import OpenAI

LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in llm.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank", "thinking"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("모델:", LLM_MODEL)

In [ ]:
# 한 번의 LLM 호출 = '에이전트 하나'. system 프롬프트로 역할을 준다.
def ask(system, user, max_tokens=500):
    return llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=0.3, max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    ).choices[0].message.content

## Lab 1 · 오케스트레이션 — 오케스트레이터-워커(팬아웃)

목표를 역할별로 쪼개 여러 워커에 맡기고(팬아웃), 결과를 합친다. 각 워커는 독립적이라 병렬이 된다.
(노트북에선 순차로 돌지만 구조는 동일 — Claude Code에선 서브에이전트로 실제 병렬.)

In [ ]:
# 오케스트레이터-워커: 목표를 역할별로 쪼개 '팬아웃'(여기선 순차 데모)하고, 합친다.
def worker(role, goal):
    return ask(f"너는 '{role}' 담당이다. 네 파트만 간결히(5줄 이내).", f"목표: {goal}")

def orchestrate(goal, roles):
    print("목표:", goal)
    parts = {r: worker(r, goal) for r in roles}                 # 팬아웃
    for r, p in parts.items():
        print(f"\n[{r}]\n{p}")
    return ask("너는 통합자다. 아래 파트들을 하나의 계획으로 합쳐 요약하라.",
               "\n\n".join(f"[{r}]\n{p}" for r, p in parts.items()))   # 합성

print("\n=== 합성 ===")
print(orchestrate("간단한 할일 관리 웹앱 만들기", ["프론트엔드", "백엔드 API", "테스트"]))

## Lab 2 · 루프 엔지니어링 — 계획→구현→검증(critic)

구현자와 **비평자를 분리**하고, 비평 결과를 되먹여 고친다. 무한 반복을 막는 **수렴 조건('OK')과 한도(rounds)**가 있다. (7강 self-correction의 확장)

In [ ]:
# 루프 엔지니어링: 계획 → 구현 → 검증(critic) → 고침 반복 (수렴 조건 + 한도).
def build_verify(task, rounds=2):
    plan = ask("너는 계획가다. 3단계 이내 계획만 제시.", task)
    print("계획:\n", plan)
    code = ask("너는 파이썬 구현자다. 코드만 제시.", f"과제: {task}\n계획:\n{plan}")
    for i in range(rounds):
        review = ask("너는 엄격한 코드 리뷰어다. 결함이 없으면 'OK'만, 있으면 결함을 나열.",
                     f"다음 코드의 결함을 찾아라:\n{code}")
        print(f"\n[검증 {i+1}] {review}")
        if review.strip().upper().startswith("OK"):            # 수렴하면 종료
            break
        code = ask("너는 파이썬 구현자다. 리뷰를 반영해 고친 코드만 제시.",
                   f"과제: {task}\n코드:\n{code}\n리뷰:\n{review}")   # 결함을 되먹여 수정
    return code

print("\n=== 최종 코드 ===")
print(build_verify("이메일 문자열이 유효한지 검사하는 함수 is_valid_email(s: str) -> bool 작성"))

## 실습 · 정적 하네스로 멀티에이전트 (Superpowers)

**입힌다** — 미리 정해진 팀 구조를 통째로 얹는다.

```
/plugin marketplace add obra/superpowers
/plugin install superpowers
```

**돌려본다** — 8강에서 다시 만든 것과 같은 과제를, 이번엔 팀으로:

```
RAG 검색 도구를 붙인 에이전트를 만들어줘. 테스트부터.
```

관찰:
- [ ] 브레인스토밍 → 스펙 → 계획 → **실패 테스트** → 구현 → 리뷰 순서가 강제되나
- [ ] **task마다 새 서브에이전트**가 뜨는가
- [ ] 리뷰가 2단계(스펙 준수 → 코드 품질)로 도나

**원리** — 정적 하네스 = **미리 정해진 팀 구조**. 내가 팀을 설계하지 않아도 프레임워크가 강제한다. 우회가 안 되는 게 장점이자 단점.

> 언제 쓰나: **품질이 어제보다 떨어질 때.**


## 실습 · 메타 하네스로 팀 구축 (GSD · harness)

**입힌다** — 팀을 그때그때 만들어 쓴다.

```powershell
npx get-shit-done-cc --claude --local

# 같은 프레임워크를 Codex에도 — 하네스만 다르다
npx get-shit-done-cc --codex
```

**돌려본다**
- 긴 작업을 주고, **스테이지가 넘어갈 때 컨텍스트가 리셋되는지** 본다.
- 8강에서 `revfactory/harness`로 **찍어낸 팀**이 있다면, 그 팀을 여기서 **실제로 돌린다**.
- 같은 과제를 Claude Code와 Codex 양쪽에서 돌려 **어디가 다른지** 본다.

**원리** — 메타 하네스 = **팀을 그때그때 만들어 쓴다.** 8강에서 *만들었고*, 9강에서 *돌린다*. 정적 하네스가 "정해진 팀"이라면 메타 하네스는 "필요한 팀".

> 언제 쓰나: **한 시간 지나면 헛소리를 할 때** (긴 작업 · 인프라).

**주의 — 토큰** — 서브에이전트를 여러 개 스폰하므로 **토큰을 크게 쓴다.** 요금제 한도를 확인하고 시작하라. 한도에 걸리면 수업 중간에 멈춘다.


## Lab 3 · 웹앱 빌드 — 멀티에이전트로 Next.js (핵심)

로컬 Claude Code에서, 여러 에이전트(서브에이전트/오케스트레이션)로 작은 Next.js 앱을 만든다.

1. **앱 정하기** — 작게. 예: 할일 관리, 메모, 간단 대시보드.
2. **인터페이스 먼저** — 페이지·API 스펙을 먼저 합의 (병렬 충돌 방지).
3. **분업** — 프론트(페이지·컴포넌트) / API(라우트) / 테스트를 나눠 위임.
4. **통합·검증** — 합친 뒤 실행하고, 검증 에이전트로 결함 리뷰 → 고침 루프.

```bash
npx create-next-app@latest my-app     # 스캐폴드
# Claude Code에서: 서브에이전트로 프론트/API/테스트를 나눠 진행
```

> 목표는 화려한 앱이 아니라 **오케스트레이션을 관찰**하는 것. 오늘 안에 브라우저에서 도는 앱 하나.

## Lab 4 · 평가와 비용 회고

- **결과 검증**: 만든 앱을 '만든 에이전트'에게 묻지 말고, 실제 실행·테스트·독립 리뷰로 확인했는가?
- **토큰 비용**: 몇 개의 에이전트가 몇 스텝을 돌았나? 병렬이 비용을 어떻게 늘렸나?
- **경계 설계**: 어디서 병렬이 부딪혔나? 무엇을 먼저 합의했어야 했나?

> (선택) 8강 미니 하네스에 '서브에이전트 스폰'을 추가해, 이 오케스트레이션을 직접 구현해봐도 좋다.

## Lab 6 · (선택) LangGraph supervisor — 프레임워크로 멀티에이전트

앞서 손으로 짠 오케스트레이션을 **LangGraph**로 하면 이렇게 됩니다: 감독(supervisor)이 워커 에이전트들에 일을 배분·종합.
`create_supervisor(agents=[...], model=...)` 한 줄이 우리 `orchestrate`의 프레임워크판이에요.

> ⚠️ **선택 랩 · 템플릿**입니다. LangGraph는 버전에 따라 API가 달라질 수 있으니, 안 되면 [LangGraph 공식 문서](https://langchain-ai.github.io/langgraph/)를 확인하세요. LLM은 Lab 0에서 만든 NVIDIA 설정을 재사용합니다.

In [ ]:
%pip install -q langgraph langgraph-supervisor langchain-openai

In [ ]:
# (선택) LangGraph supervisor 템플릿 — Lab 0의 LLM_BASE_URL/NVIDIA_API_KEY/LLM_MODEL 재사용
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

llm_lc = ChatOpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY, model=LLM_MODEL, temperature=0.2)

def add(a: int, b: int) -> int:
    """두 정수를 더한다"""
    return a + b

# 워커 에이전트 두 명 (각자 역할)
math_agent = create_react_agent(llm_lc, tools=[add], name="math", prompt="너는 계산 담당이다.")
writer_agent = create_react_agent(llm_lc, tools=[], name="writer", prompt="너는 요약 담당이다.")

# 감독이 배분·종합
app = create_supervisor(
    agents=[math_agent, writer_agent], model=llm_lc,
    prompt="너는 감독이다. 계산은 math, 글쓰기는 writer 에게 맡겨라.",
).compile()

result = app.invoke({"messages": [{"role": "user", "content": "3 더하기 4를 계산하고 한 줄로 요약해줘"}]})
print(result["messages"][-1].content)

## 산출물 체크리스트

- [ ] 오케스트레이션 패턴을 손으로 짜봤다 — 오케스트레이터-워커(팬아웃)
- [ ] 루프 엔지니어링을 손으로 짜봤다 — 계획 → 구현 → 검증(critic) → 반복
- [ ] **정적 하네스(Superpowers)를 설치해 팀으로 돌렸다** — task마다 새 서브에이전트
- [ ] **메타 하네스(GSD)를 설치해 돌렸다** — 스테이지마다 신선한 컨텍스트
- [ ] **GSD를 Codex에도 설치해** 같은 프레임워크가 다른 하네스에서 도는 걸 봤다
- [ ] 8강에서 찍어낸 팀(harness)이 있다면 **여기서 실제로 돌렸다**
- [ ] 멀티에이전트로 **웹앱을 빌드**했다 (Lab 3)
- [ ] 결과를 **독립 검증**하고 토큰·비용을 회고했다

> 한 줄: **파이썬으로 원리를 이해하고, 실전 하네스로 규모를 얻는다.**
